# Titanic exp012 - Replace LogisticRegression with RandomForest on top of Title + FamilyGroup + SexPclass

この notebook は現在のベスト構成 `Title + FamilyGroup + SexPclass` の特徴量をそのまま使い、モデルだけを `RandomForest` に置き換えて比較するためのものです。

仮説:
- `Title`、`FamilyGroup`、`SexPclass` の組み合わせには非線形な分岐が残っているかもしれない
- 線形モデルよりも `RandomForest` の方がより柔軟に表現できるかを確認する


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp012_randomforest.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "FamilyGroup", "SexPclass"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass"]

TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}


In [2]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out["Name"].fillna("").astype(str).str.extract(r",\\s*([^\\.]+)\\.", expand=False)
    out["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    out["FamilySize"] = out["SibSp"].fillna(0) + out["Parch"].fillna(0) + 1
    out["FamilyGroup"] = pd.cut(
        out["FamilySize"],
        bins=[0, 1, 4, 100],
        labels=["alone", "small", "large"],
        right=True,
    ).astype(str)
    out["SexPclass"] = out["Sex"].fillna("missing").astype(str) + "_" + out["Pclass"].fillna(-1).astype(int).astype(str)
    return out


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [3]:
train_df = add_features(pd.read_csv(TRAIN_PATH))
test_df = add_features(pd.read_csv(TEST_PATH))

display(train_df[["Sex", "Pclass", "SexPclass", "Title", "FamilyGroup"]].head())


      Sex  Pclass SexPclass Title FamilyGroup
0    male       3    male_3    Mr       small
1  female       1  female_1   Mrs       small
2  female       3  female_3  Miss       alone
3  female       1  female_1   Mrs       small
4    male       3    male_3    Mr       alone

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            NUMERIC_COLUMNS,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", make_one_hot_encoder()),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)


In [5]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))


CV scores: [0.81006, 0.79775, 0.79775, 0.82022, 0.81461]
CV mean: 0.80808
CV std: 0.00903


比較メモ:

- `exp007` Title + FamilyGroup + SexPclass CV mean: `0.83614`
- `exp012` exp007 features + RandomForest CV mean: `0.80808`
- Difference vs `exp007`: `-0.02806`


In [6]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())


saved: /home/sora/dev/kaggle/competitions/titanic/submissions/exp012_randomforest.csv


   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         1
4          896         0